<!-- codex_annotation: script_overview -->
# 基础模板匹配坐标生成

读取完整 DAPI 图和 tile 文件夹，对每个 tile 做全局模板匹配，输出 TileConfiguration.txt。

注释说明：
- 适合 tile 数量较少或参考图不太大的情况。
- 大图很大时建议使用 02.merge_dapi.ipynb 的粗匹配加精匹配流程。


In [2]:
import os
import cv2
import tifffile as tiff
import numpy as np


In [3]:
#==========================
# 路径设置
# ==========================

FULL_IMAGE = r"D:\01.analysis\11.test_result\MAX_C1-P4-rep2-A.tif"

TILE_DIR = r"D:\01.analysis\11.test_result\DAPI"

OUTPUT_FILE = r"D:\01.analysis\11.test_result\DAPI\TileConfiguration.txt"


In [4]:
# ==========================
# 读取完整脑片
# ==========================

print("Loading full brain...")

full = tiff.imread(FULL_IMAGE)

if full.ndim > 2:
    full = full[0]

full = cv2.normalize(
    full,
    None,
    0,
    255,
    cv2.NORM_MINMAX
).astype(np.uint8)

print("Full brain size:", full.shape)

Loading full brain...
Full brain size: (11120, 17303)


In [5]:
# ==========================
# Tile列表
# ==========================

tile_files = sorted([
    f for f in os.listdir(TILE_DIR)
    if f.endswith(".tif")
])

results = []

In [6]:
# ==========================
# 搜索Tile位置
# ==========================

for tile_name in tile_files:

    print("\nSearching:", tile_name)

    tile_path = os.path.join(
        TILE_DIR,
        tile_name
    )

    tile = tiff.imread(tile_path)

    if tile.ndim > 2:
        tile = tile[0]

    tile = cv2.normalize(
        tile,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    ).astype(np.uint8)

    print(
        "Tile size:",
        tile.shape
    )

    result = cv2.matchTemplate(
        full,
        tile,
        cv2.TM_CCOEFF_NORMED
    )

    _, max_val, _, max_loc = cv2.minMaxLoc(result)

    x, y = max_loc

    print(
        f"Position = ({x},{y}) "
        f"Score={max_val:.3f}"
    )

    results.append(
        (
            tile_name,
            x,
            y
        )
    )



Searching: tile_01.tif
Tile size: (6419, 2845)
Position = (9477,1546) Score=0.398

Searching: tile_02.tif
Tile size: (6420, 2852)
Position = (3721,13) Score=0.249

Searching: tile_04.tif
Tile size: (6427, 3739)
Position = (8768,1799) Score=0.732

Searching: tile_05.tif


: 

In [ ]:
# ==========================
# 输出Fiji坐标文件
# ==========================

with open(
    OUTPUT_FILE,
    "w"
) as f:

    f.write("dim = 2\n\n")

    for tile_name,x,y in results:

        f.write(
            f"{tile_name}; ; ({x},{y})\n"
        )

print("\nSaved:")
print(OUTPUT_FILE)